In [1]:
# ==========================================
# PASO 1: CARGA Y DIAGNÓSTICO
# ==========================================
import pandas as pd

def miles(n):
    """Formatea un entero con punto como separador de miles (ej: 103904 -> 103.904)."""
    return f"{n:,}".replace(",", ".")

url = "https://raw.githubusercontent.com/egle06/airline-satisfaction-analysis/refs/heads/main/train.csv"
df = pd.read_csv(url, index_col=0)

# 1. Estructura y tipos
print(f"Filas: {miles(df.shape[0])} | Columnas: {df.shape[1]}\n")
df.info()

# 2. Estadísticas descriptivas
print("\n--- describe() ---")
display(df.describe().T)

# 3. Nulos por columna
print("\n--- Nulos por columna ---")
nulos = df.isnull().sum()
display(nulos[nulos > 0])

# 4. Duplicados: filas completas e id repetidos
print(f"\nFilas completamente duplicadas: {df.duplicated().sum()}")
print(f"Filas con id repetido: {df['id'].duplicated().sum()}")

# 5. Consistencia de las variables categóricas
print("\n--- Categorías de las variables de texto ---")
for col in ['Gender', 'Customer Type', 'Type of Travel', 'Class', 'satisfaction']:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))

Filas: 103.904 | Columnas: 24

<class 'pandas.core.frame.DataFrame'>
Index: 103904 entries, 0 to 103903
Data columns (total 24 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   id                                 103904 non-null  int64  
 1   Gender                             103904 non-null  object 
 2   Customer Type                      103904 non-null  object 
 3   Age                                103904 non-null  int64  
 4   Type of Travel                     103904 non-null  object 
 5   Class                              103904 non-null  object 
 6   Flight Distance                    103904 non-null  int64  
 7   Inflight wifi service              103904 non-null  int64  
 8   Departure/Arrival time convenient  103904 non-null  int64  
 9   Ease of Online booking             103904 non-null  int64  
 10  Gate location                      103904 non-null  int64  
 11  Food and drin

,count,mean,std,min,25%,50%,75%,max
id,103904.0,64924.210502,37463.812252,1.0,32533.75,64856.5,97368.25,129880.0
Age,103904.0,39.379706,15.114964,7.0,27.00,40.0,51.00,85.0
Flight Distance,103904.0,1189.448375,997.147281,31.0,414.00,843.0,1743.00,4983.0
Inflight wifi service,103904.0,2.729683,1.327829,0.0,2.00,3.0,4.00,5.0
Departure/Arrival time convenient,103904.0,3.060296,1.525075,0.0,2.00,3.0,4.00,5.0
Ease of Online booking,103904.0,2.756901,1.398929,0.0,2.00,3.0,4.00,5.0
Gate location,103904.0,2.976883,1.277621,0.0,2.00,3.0,4.00,5.0
Food and drink,103904.0,3.202129,1.329533,0.0,2.00,3.0,4.00,5.0
Online boarding,103904.0,3.250375,1.349509,0.0,2.00,3.0,4.00,5.0
Seat comfort,103904.0,3.439396,1.319088,0.0,2.00,4.0,5.00,5.0



--- Nulos por columna ---


,0
Arrival Delay in Minutes,310



Filas completamente duplicadas: 0
Filas con id repetido: 0

--- Categorías de las variables de texto ---

Gender:
Gender
Female    52727
Male      51177
Name: count, dtype: int64

Customer Type:
Customer Type
Loyal Customer       84923
disloyal Customer    18981
Name: count, dtype: int64

Type of Travel:
Type of Travel
Business travel    71655
Personal Travel    32249
Name: count, dtype: int64

Class:
Class
Business    49665
Eco         46745
Eco Plus     7494
Name: count, dtype: int64

satisfaction:
satisfaction
neutral or dissatisfied    58879
satisfied                  45025
Name: count, dtype: int64


In [2]:
# ==========================================
# PASO 2A: ¿CUÁNTOS CEROS HAY EN LAS ESCALAS DE SERVICIO?
# ==========================================
cols_servicio = ['Inflight wifi service', 'Departure/Arrival time convenient',
                 'Ease of Online booking', 'Gate location', 'Food and drink',
                 'Online boarding', 'Seat comfort', 'Inflight entertainment',
                 'On-board service', 'Leg room service', 'Baggage handling',
                 'Checkin service', 'Inflight service', 'Cleanliness']

ceros = pd.DataFrame({
    'Ceros': (df[cols_servicio] == 0).sum(),
    'Porcentaje (%)': ((df[cols_servicio] == 0).mean() * 100).round(2)
}).sort_values('Ceros', ascending=False)

display(ceros)
print(f"Filas con al menos un 0 en alguna escala: {miles((df[cols_servicio] == 0).any(axis=1).sum())}")


,Ceros,Porcentaje (%)
Departure/Arrival time convenient,5300,5.10
Ease of Online booking,4487,4.32
Inflight wifi service,3103,2.99
Online boarding,2428,2.34
Leg room service,472,0.45
Food and drink,107,0.10
Inflight entertainment,14,0.01
Cleanliness,12,0.01
Inflight service,3,0.00
On-board service,3,0.00


Filas con al menos un 0 en alguna escala: 8.200


In [6]:
# ==========================================
# PASO 2B: LIMPIEZA
# ==========================================
# Trabajamos sobre una copia: df queda como dataset original
df_limpio = df.copy()
print(f"Filas iniciales: {miles(len(df_limpio))}")

# --- 1. DUPLICADOS (verificado: 0, se deja la instrucción por robustez) ---
df_limpio = df_limpio.drop_duplicates(subset='id')
print(f"Tras eliminar duplicados: {miles(len(df_limpio))} filas")

# --- 2. NORMALIZACIÓN DE CATEGORÍAS (mayúsculas inconsistentes) ---
for col in ['Customer Type', 'Type of Travel']:
    df_limpio[col] = df_limpio[col].str.strip().str.title()

print("\nCategorías tras normalizar:")
print(df_limpio['Customer Type'].value_counts())
print(df_limpio['Type of Travel'].value_counts())

# --- 3. IMPUTACIÓN DE NULOS: mediana (la variable es muy asimétrica, la media no es representativa) ---
mediana_llegada = df_limpio['Arrival Delay in Minutes'].median()
df_limpio['Arrival Delay in Minutes'] = df_limpio['Arrival Delay in Minutes'].fillna(mediana_llegada)
print(f"\nNulos imputados con la mediana ({mediana_llegada} min). Nulos restantes: {df_limpio['Arrival Delay in Minutes'].isnull().sum()}")

# --- 4. CONVERSIÓN DE TIPOS ---
df_limpio['Arrival Delay in Minutes'] = df_limpio['Arrival Delay in Minutes'].astype(int)  # minutos enteros
for col in ['Gender', 'Customer Type', 'Type of Travel', 'Class', 'satisfaction']:
    df_limpio[col] = df_limpio[col].astype('category')

# --- 5. RESET DEL ÍNDICE ---
df_limpio = df_limpio.reset_index(drop=True)

print("\n--- Tipos de datos tras la limpieza ---")
print(df_limpio.dtypes.value_counts())
print(f"Nulos totales: {df_limpio.isnull().sum().sum()}")

Filas iniciales: 103.904
Tras eliminar duplicados: 103.904 filas

Categorías tras normalizar:
Customer Type
Loyal Customer       84923
Disloyal Customer    18981
Name: count, dtype: int64
Type of Travel
Business Travel    71655
Personal Travel    32249
Name: count, dtype: int64

Nulos imputados con la mediana (0.0 min). Nulos restantes: 0

--- Tipos de datos tras la limpieza ---
int64       19
category     1
category     1
category     1
category     1
category     1
Name: count, dtype: int64
Nulos totales: 0


In [8]:
# ==========================================
# PASO 2C: TRATAMIENTO DE CEROS EN ESCALAS DE SERVICIO
# ==========================================
# Los 0 se interpretan como "no aplica / sin respuesta": se convierten a NaN
# y se imputan con la moda (valor más frecuente), porque son escalas ordinales enteras.

cols_servicio = ['Inflight wifi service', 'Departure/Arrival time convenient',
                 'Ease of Online booking', 'Gate location', 'Food and drink',
                 'Online boarding', 'Seat comfort', 'Inflight entertainment',
                 'On-board service', 'Leg room service', 'Baggage handling',
                 'Checkin service', 'Inflight service', 'Cleanliness']

total_ceros = (df_limpio[cols_servicio] == 0).sum().sum()
print(f"Ceros a tratar: {miles(total_ceros)}")

for col in cols_servicio:
    moda = df_limpio[col][df_limpio[col] != 0].mode()[0]   # moda calculada sin contar los ceros
    df_limpio[col] = df_limpio[col].mask(df_limpio[col] == 0, moda)

# Verificación
print(f"Ceros restantes: {(df_limpio[cols_servicio] == 0).sum().sum()}")
print(f"Nulos restantes: {df_limpio[cols_servicio].isnull().sum().sum()}")
print(f"Rango de las escalas: {df_limpio[cols_servicio].min().min()} a {df_limpio[cols_servicio].max().max()}")

Ceros a tratar: 15.932
Ceros restantes: 0
Nulos restantes: 0
Rango de las escalas: 1 a 5


In [9]:
# ==========================================
# VERIFICACIÓN FINAL DE LA LIMPIEZA
# ==========================================
print(f"Filas: {miles(df_limpio.shape[0])} | Columnas: {df_limpio.shape[1]}")
print(f"Nulos totales: {df_limpio.isnull().sum().sum()}")
print(f"Duplicados: {df_limpio.duplicated().sum()}")
print(f"Índice: de {df_limpio.index.min()} a {miles(df_limpio.index.max())}\n")
df_limpio.info()

Filas: 103.904 | Columnas: 24
Nulos totales: 0
Duplicados: 0
Índice: de 0 a 103.903

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103904 entries, 0 to 103903
Data columns (total 24 columns):
 #   Column                             Non-Null Count   Dtype   
---  ------                             --------------   -----   
 0   id                                 103904 non-null  int64   
 1   Gender                             103904 non-null  category
 2   Customer Type                      103904 non-null  category
 3   Age                                103904 non-null  int64   
 4   Type of Travel                     103904 non-null  category
 5   Class                              103904 non-null  category
 6   Flight Distance                    103904 non-null  int64   
 7   Inflight wifi service              103904 non-null  int64   
 8   Departure/Arrival time convenient  103904 non-null  int64   
 9   Ease of Online booking             103904 non-null  int64   
 10  Gate lo

In [10]:
# ==========================================
# PASO 3: AGREGACIONES DE NEGOCIO
# ==========================================
# Columna auxiliar: 1 si el pasajero está satisfecho, 0 si no (para poder promediar)
df_limpio['satisfecho'] = (df_limpio['satisfaction'] == 'satisfied').astype(int)

# 1. ¿Qué % de pasajeros satisfechos hay según clase y tipo de viaje?
print("1. % de satisfechos por Clase y Tipo de viaje")
g1 = (df_limpio.groupby(['Class', 'Type of Travel'], observed=True)['satisfecho']
      .agg(['mean', 'count']))
g1['mean'] = (g1['mean'] * 100).round(1)
display(g1.rename(columns={'mean': '% satisfechos', 'count': 'pasajeros'}))

# 2. ¿Difieren las valoraciones de wifi y asiento entre satisfechos y no satisfechos?
print("\n2. Valoración promedio de wifi y asiento según satisfacción")
display(df_limpio.groupby('satisfaction', observed=True)[['Inflight wifi service', 'Seat comfort']].mean().round(2))

# 3. ¿Cómo son las demoras según satisfacción? (mediana además de media, por los outliers)
print("\n3. Demoras (minutos) según satisfacción")
display(df_limpio.groupby('satisfaction', observed=True)[['Departure Delay in Minutes', 'Arrival Delay in Minutes']]
        .agg(['mean', 'median']).round(1))

# 4. ¿Cuántos pasajeros y qué % de satisfechos hay por tipo de cliente?
print("\n4. Satisfacción por tipo de cliente")
g4 = df_limpio.groupby('Customer Type', observed=True)['satisfecho'].agg(['count', 'mean'])
g4['mean'] = (g4['mean'] * 100).round(1)
display(g4.rename(columns={'count': 'pasajeros', 'mean': '% satisfechos'}))

1. % de satisfechos por Clase y Tipo de viaje


% satisfechos  pasajeros
Class    Type of Travel                           
Business Business Travel           72.0      47508
         Personal Travel           12.2       2157
Eco      Business Travel           29.6      20257
         Personal Travel           10.2      26488
Eco Plus Business Travel           39.3       3890
         Personal Travel            8.7       3604


2. Valoración promedio de wifi y asiento según satisfacción


,Inflight wifi service,Seat comfort
satisfaction,,
neutral or dissatisfied,2.40,3.04
satisfied,3.37,3.97



3. Demoras (minutos) según satisfacción


Departure Delay in Minutes         \
                                              mean median   
satisfaction                                                
neutral or dissatisfied                       16.5    0.0   
satisfied                                     12.6    0.0   

                        Arrival Delay in Minutes         
                                            mean median  
satisfaction                                             
neutral or dissatisfied                     17.1    0.0  
satisfied                                   12.6    0.0


4. Satisfacción por tipo de cliente


,pasajeros,% satisfechos
Customer Type,,
Disloyal Customer,18981,23.7
Loyal Customer,84923,47.7


### Interpretación de las agrupaciones

1. **Clase y tipo de viaje:** es el factor con mayor diferencia. En viajes de negocios, el 72 % de los pasajeros de clase Business está satisfecho, frente al 29,6 % en Eco y 39,3 % en Eco Plus. En viajes personales la satisfacción es baja en las tres clases (8,7 % a 12,2 %). El grupo Business + Personal Travel es reducido (2.157 pasajeros), por lo que su porcentaje es menos estable.
2. **Wifi y asiento:** los pasajeros satisfechos puntúan casi un punto más alto en ambos servicios (wifi: 3,37 vs 2,40; asiento: 3,97 vs 3,04), lo que respalda la hipótesis de que son factores relevantes.
3. **Demoras:** las medias difieren poco (≈ 4 minutos) y la mediana es 0 en ambos grupos. Con este análisis descriptivo, las demoras no parecen un factor diferenciador fuerte.
4. **Tipo de cliente:** los clientes leales tienen una satisfacción del 47,7 %, el doble que los desleales (23,7 %).

**Conclusión preliminar:** la hipótesis se sostiene parcialmente. Wifi y asiento muestran diferencias claras, pero las demoras casi no se diferencian, y la clase combinada con el tipo de viaje pesa más de lo supuesto. Estas son asociaciones descriptivas, no relaciones causales; la importancia relativa de cada variable se evaluará con modelos en etapas posteriores.